# Indicator 5: Fibonacci Retracement

**Core idea:** Find the most recent swing high and swing low over a lookback window. Compute the key Fibonacci retracement levels (23.6%, 38.2%, 50%, 61.8%) between those two extremes. Generate a signal based on where the current price sits relative to those levels.

| Zone | Interpretation | Signal |
|------|---------------|--------|
| Near 38.2% or 61.8% retracement from a **swing low** | Price pulling back to support — potential bounce | +1 (Buy) |
| Near 38.2% or 61.8% retracement from a **swing high** | Price rallying into resistance — potential fade | −1 (Sell) |
| Elsewhere | No actionable level nearby | 0 (Hold) |

The `proximity_pct` tolerance (default 1.5%) defines how close the price must be to a level to count as "near" it.

In [2]:
# ---- Indicator 5: Fibonacci Retracement ------------------------------------
#
# Steps:
#   1. Over the last `window` bars, find the swing high (max price) and
#      swing low (min price).
#   2. Determine the dominant move direction:
#        - If swing high occurred *after* swing low  -> uptrend:
#          retracement levels are pulled back *down* from the high.
#        - If swing low occurred *after* swing high  -> downtrend:
#          retracement levels are pulled back *up* from the low.
#   3. Compute the four standard Fibonacci retracement levels.
#   4. Check whether the current price is within `proximity_pct` of the
#      two key levels (38.2% and 61.8%).
#        - In an uptrend, those levels are support -> signal = +1
#        - In a downtrend, those levels are resistance -> signal = -1

FIBO_LEVELS = [0.236, 0.382, 0.500, 0.618]   # standard retracement ratios
KEY_LEVELS  = {0.382, 0.618}                  # levels that generate a signal


def _fib_retracement_levels(swing_low, swing_high):
    '''Return a dict {ratio: price_level} for the four Fibonacci levels.
    Levels are measured from swing_low up toward swing_high.
    A 0.382 level means price has retraced 38.2% of the full move.
    '''
    move = swing_high - swing_low
    return {r: swing_high - r * move for r in FIBO_LEVELS}


def indicator_5_fibonacci(prices_series, date_idx, window=60, proximity_pct=0.015):
    '''Fibonacci Retracement signal.

    Parameters
    ----------
    prices_series  : list of floats — full price history up to current bar
    date_idx       : int            — current bar index (0-based)
    window         : int            — lookback for swing high / low (default 60)
    proximity_pct  : float          — how close price must be to a level,
                                      as a fraction of current price (default 1.5%)

    Returns
    -------
     1  Price is near a Fibonacci support level (38.2% or 61.8% in uptrend)
    -1  Price is near a Fibonacci resistance level (38.2% or 61.8% in downtrend)
     0  No key Fibonacci level nearby, or insufficient data
    '''
    if date_idx < window:
        return 0

    window_prices = prices_series[date_idx - window + 1:date_idx + 1]
    current_price = prices_series[date_idx]

    swing_high = max(window_prices)
    swing_low  = min(window_prices)

    # Avoid flat / near-flat price series
    if swing_high - swing_low < 1e-9:
        return 0

    high_idx = window_prices.index(swing_high)
    low_idx  = window_prices.index(swing_low)

    # Dominant direction: whichever extreme came last sets the context
    uptrend = high_idx > low_idx   # high formed after low -> uptrend

    levels = _fib_retracement_levels(swing_low, swing_high)

    tolerance = current_price * proximity_pct

    for ratio, level_price in levels.items():
        if ratio not in KEY_LEVELS:
            continue
        if abs(current_price - level_price) <= tolerance:
            return 1 if uptrend else -1

    return 0